# GrantScopeAI — Source Access and Validation

## Project Context
Researchers and research-support teams often search grant databases, funder websites, and publication indexes separately. This makes it difficult to see how a topic is funded across regions, which institutions are active, whether research activity is growing, and which funded projects are most comparable to a new concept.

GrantScopeAI consolidates these signals into one exploratory research-intelligence interface.

### Intended User

The primary intended user is a researcher or university research-support professional assessing how to position an early grant concept.

Potential users include:

- Researchers and principal investigators
- University grant offices
- Research-support professionals
- Research strategy teams
- Scientific consultancies

### Primary User Story

As a researcher developing an AI-enabled chemistry or materials proposal, I want to compare my concept with recent funded projects and publication trends so that I can identify relevant funders, refine my positioning, and document the evidence behind my choices.

### Core Decision Supported

GrantScopeAI is intended to help users answer:

Where does a proposed AI-enabled chemistry or materials topic fit within recent funding and publication activity?

### Primary Research Question

How can public grant and publication data help a researcher identify funding patterns, comparable funded projects, and research momentum for an AI-enabled chemistry or materials proposal?

### Supporting Questions

1. How have grant counts and reported award amounts changed over time?
2. Which funders, programmes, countries, and organisations are most active?
3. Which scientific topics appear most frequently in funded projects?
4. Which previously funded projects are most similar to a new proposal concept?
5. Is publication activity increasing or decreasing for selected topics?
6. What data-quality and comparability limitations affect the analysis?

### Initial Scientific Scope

The initial project scope focuses on AI-enabled chemistry and materials research, including:

- Catalysis
- Molecular modelling
- Reaction prediction
- Materials discovery
- Laboratory automation
- Scientific machine learning

The initial analysis period is 2021–2025. Records from 2026 may be included only when coverage is sufficiently complete and clearly labelled as partial-year data.

### Data Source Roles

The project uses three public data sources:

- **CORDIS:** European Union-funded research projects and grant information
- **NSF Award Search:** United States research awards and funding metadata
- **OpenAlex:** Publication activity, research topics, institutions, countries, and citation context

CORDIS and NSF records will be standardised into a common grants dataset. OpenAlex publications will remain in a separate dataset and will be used to provide aggregate research-momentum context.

The project will not force direct row-level matches between grants and publications unless a reliable relationship can be established.

### Planned Project Output

GrantScopeAI will provide:

- Funding-trend exploration
- Analysis of active funders, programmes, organisations, and countries
- Topic-level funding and publication comparisons
- Data-quality reporting
- A keyword-overlap recommendation baseline
- A TF-IDF and cosine-similarity recommender for finding similar funded projects
- A four-page Streamlit decision-support application

### Project Boundaries

GrantScopeAI is an exploratory research-intelligence prototype. It will not:

- Write complete grant proposals
- Determine whether a research idea is genuinely novel
- Predict whether a proposal will receive funding
- Replace official funder eligibility checks
- Establish causal relationships between funding and publication growth
- Compare EUR and USD award totals without a documented conversion method

# Source validation and Raw Dataset Creation
## ### Initial NSF API Test

In [1]:
import json
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

print(f"Extraction date: {EXTRACTION_DATE}")
print(f"Raw data folder: {RAW_DATA_DIR.resolve()}")

Extraction date: 2026-08-01
Raw data folder: C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw


In [2]:
def inspect_api_response(response, source_name):
    """
    Display basic validation information for an API response.
    """
    print(f"Source: {source_name}")
    print(f"Status code: {response.status_code}")
    print(f"Content type: {response.headers.get('content-type')}")
    print(f"Response size: {len(response.content):,} bytes")

    response.raise_for_status()

In [3]:
NSF_API_URL = "https://api.nsf.gov/services/v1/awards.json"

nsf_params = {
    "keyword": "machine learning chemistry",
    "startDateStart": "01/01/2021",
    "startDateEnd": "12/31/2025",
    "offset": 1,
    "printFields": ",".join([
        "id",
        "title",
        "abstractText",
        "startDate",
        "expDate",
        "awardeeName",
        "awardeeCity",
        "awardeeStateCode",
        "awardeeCountryCode",
        "fundsObligatedAmt",
        "estimatedTotalAmt",
        "fundProgramName",
        "primaryProgram",
        "agency"
    ])
}

nsf_response = requests.get(
    NSF_API_URL,
    params=nsf_params,
    timeout=30
)

inspect_api_response(nsf_response, "NSF Award Search")

Source: NSF Award Search
Status code: 200
Content type: application/json
Response size: 127,271 bytes


In [4]:
nsf_json = nsf_response.json()

print(nsf_json.keys())
print(nsf_json.get("response", {}).keys())

dict_keys(['response'])
dict_keys(['award', 'metadata'])


In [5]:
nsf_awards = nsf_json.get("response", {}).get("award", [])

print(f"Number of awards returned: {len(nsf_awards)}")

nsf_df = pd.DataFrame(nsf_awards)
nsf_df.head()

Number of awards returned: 25


,abstractText,activeAwd,agency,awardAgencyCode,awardee,awardeeAddress,awardeeCity,awardeeCountryCode,awardeeDistrict,awardeeDistrictCode,...,program,progRefCode,publicAccessMandate,startDate,title,transType,ueiNumber,coPDPI,jrnl,publicationResearch
0,Beaches are coastline features that offer econ...,true,NSF,4900,FLORIDA INTERNATIONAL UNIVERSITY,11200 SW 8TH ST,MIAMI,US,26,FL26,...,RET SUPP-Res Exp for Tchr Supp,7218,1,12/15/2025,Collaborative Research: Swash zone dynamics dr...,Standard Grant,Q3KCVK5S9CP1,NaN,NaN,NaN
1,This project aims to serve the national intere...,true,NSF,4900,"LOYOLA UNIVERSITY MARYLAND, INC.",4501 N CHARLES ST,BALTIMORE,US,02,MD02,...,"QUANTUM INFORMATION SCIENCE, Improv Undergrad ...","7203, 8209, 9178",1,12/15/2025,Cornerstones for an Undergraduate Quantum Comp...,Standard Grant,FV5AVEGVTUE4,[Mary L Lowe mlowe@loyola.edu],NaN,NaN
2,"Coral reefs nurture fisheries, protect coastli...",true,NSF,4900,"UNIVERSITY OF CALIFORNIA, LOS ANGELES",10889 WILSHIRE BLVD STE 700,LOS ANGELES,US,36,CA36,...,,,1,12/15/2025,Collaborative Research: BoCP-Implementation: A...,Standard Grant,RN64EPNH8JC6,"[George Perry ghp3@psu.edu, Laura S Weyrich ls...",NaN,NaN
3,"Coral reefs nurture fisheries, protect coastli...",true,NSF,4900,REGENTS OF THE UNIVERSITY OF MICHIGAN,1109 GEDDES AVE STE 3300,ANN ARBOR,US,06,MI06,...,,,1,12/15/2025,Collaborative Research: BoCP-Implementation: A...,Standard Grant,GNJ7BBP73WE9,NaN,NaN,NaN
4,Following the collapse of a cloud core to form...,true,NSF,4900,PRESIDENT AND FELLOWS OF HARVARD COLLEGE,1033 MASSACHUSETTS AVE STE 3,CAMBRIDGE,US,05,MA05,...,LABORATORY ASTROPHYSICS,1205,1,12/15/2025,Survival of Interstellar Organics in Protoplan...,Standard Grant,LN53LCFJFL45,NaN,NaN,NaN


In [26]:
print(sorted(nsf_sample_df.columns.tolist()))

['abstractText', 'activeAwd', 'agency', 'awardAgencyCode', 'awardee', 'awardeeAddress', 'awardeeCity', 'awardeeCountryCode', 'awardeeDistrict', 'awardeeDistrictCode', 'awardeeName', 'awardeePhone', 'awardeeStateCode', 'awardeeZipCode', 'cfdaNumber', 'coPDPI', 'date', 'dirAbbr', 'divAbbr', 'estimatedTotalAmt', 'expDate', 'fundAgencyCode', 'fundProgramName', 'fundsObligated', 'fundsObligatedAmt', 'histAwd', 'id', 'initAmendmentDate', 'jrnl', 'latestAmendmentDate', 'managingPec', 'orgCodeDir', 'orgCodeDiv', 'orgLongName', 'orgLongName2', 'orgUrl', 'parentUeiNumber', 'pdPIName', 'perfAddress', 'perfCity', 'perfCountryCode', 'perfDistrict', 'perfDistrictCode', 'perfLocation', 'perfStateCode', 'perfZipCode', 'pi', 'piEmail', 'piFirstName', 'piId', 'piLastName', 'piMiddeInitial', 'poEmail', 'poName', 'poPhone', 'primaryProgram', 'progEleCode', 'progRefCode', 'program', 'publicAccessMandate', 'publicationResearch', 'startDate', 'title', 'transType', 'ueiNumber']


# NSF info Extraction

In [13]:
NSF_QUERIES = [
    '"machine learning" AND chemistry',
    '"artificial intelligence" AND chemistry',
    '"deep learning" AND chemistry',
    '"machine learning" AND materials',
    '"artificial intelligence" AND materials',
    '"deep learning" AND materials',
    '"machine learning" AND catalysis',
    '"materials informatics"',
    "cheminformatics",
    '"molecular machine learning"',
    '"reaction prediction"',
    '"autonomous laboratory"',
    '"self-driving laboratory"'
]

START_DATE = date(2021, 1, 1)
END_DATE = date(2025, 12, 31)

RAW_DATA_DIR = Path("../data/raw/nsf")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

print("NSF queries:")
for query in NSF_QUERIES:
    print("-", query)

print("Date range:", START_DATE, "to", END_DATE)
print("Save location:", RAW_DATA_DIR.resolve())

NSF queries:
- "machine learning" AND chemistry
- "artificial intelligence" AND chemistry
- "deep learning" AND chemistry
- "machine learning" AND materials
- "artificial intelligence" AND materials
- "deep learning" AND materials
- "machine learning" AND catalysis
- "materials informatics"
- cheminformatics
- "molecular machine learning"
- "reaction prediction"
- "autonomous laboratory"
- "self-driving laboratory"
Date range: 2021-01-01 to 2025-12-31
Save location: C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf


In [9]:
NSF_QUERIES = [
    "machine learning chemistry",
    "artificial intelligence chemistry",
    "deep learning chemistry",
    "machine learning materials",
    "artificial intelligence materials",
    "deep learning materials",
    "materials informatics",
    "cheminformatics",
    "molecular machine learning",
    "reaction prediction",
    "machine learning catalysis",
    "autonomous laboratory",
    "self-driving laboratory"
]

In [14]:
def generate_monthly_windows(start_date, end_date):
    """
    Generate monthly date ranges between two dates.
    """
    current_date = date(
        start_date.year,
        start_date.month,
        1
    )

    while current_date <= end_date:
        final_day = monthrange(
            current_date.year,
            current_date.month
        )[1]

        window_end = date(
            current_date.year,
            current_date.month,
            final_day
        )

        window_start = max(current_date, start_date)
        window_end = min(window_end, end_date)

        yield window_start, window_end

        if current_date.month == 12:
            current_date = date(
                current_date.year + 1,
                1,
                1
            )
        else:
            current_date = date(
                current_date.year,
                current_date.month + 1,
                1
            )

In [15]:
def fetch_nsf_awards_for_period(
    start_date,
    end_date,
    keyword= None,
    delay= 0.5
):
    """
    Retrieve all NSF awards whose project start date falls
    within the specified period.
    """
    period_awards = []
    offset = 0
    results_per_page = 25

    while True:
        params = {
            "startDateStart": start_date.strftime("%m/%d/%Y"),
            "startDateEnd": end_date.strftime("%m/%d/%Y"),
            "rpp": results_per_page,
            "offset": offset,
            "printFields": ",".join([
                "id",
                "title",
                "abstractText",
                "date",
                "startDate",
                "expDate",
                "awardeeName",
                "awardeeCity",
                "awardeeStateCode",
                "awardeeCountryCode",
                "fundsObligatedAmt",
                "estimatedTotalAmt",
                "fundProgramName",
                "primaryProgram",
                "agency",
                "piFirstName",
                "piLastName",
                "dirAbbr",
                "divAbbr"
            ])
        }

        if keyword:
            params["keyword"] = keyword

        response = requests.get(
            NSF_API_URL,
            params=params,
            timeout=60
        )

        response.raise_for_status()
        payload = response.json()

        response_data = payload.get("response", {})
        awards = response_data.get("award", []) or []

        if isinstance(awards, dict):
            awards = [awards]

        period_awards.extend(awards)

        print(
            f"{start_date:%Y-%m}: "
            f"page returned {len(awards)}, "
            f"total collected {len(period_awards)}"
        )

        if len(awards) < results_per_page:
            break

        offset += results_per_page
        time.sleep(delay)

    return period_awards

In [16]:
query_test_results = []

for query in NSF_QUERIES:
    test_awards = fetch_nsf_awards_for_period(
        start_date=date(2021, 1, 1),
        end_date=date(2021, 1, 31),
        keyword=query
    )

    query_test_results.append({
        "query": query,
        "records_returned": len(test_awards)
    })

query_test_df = pd.DataFrame(query_test_results)

display(query_test_df)

2021-01: page returned 7, total collected 7
2021-01: page returned 2, total collected 2
2021-01: page returned 1, total collected 1
2021-01: page returned 22, total collected 22
2021-01: page returned 12, total collected 12
2021-01: page returned 6, total collected 6
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0
2021-01: page returned 0, total collected 0


,query,records_returned
0,"""machine learning"" AND chemistry",7
1,"""artificial intelligence"" AND chemistry",2
2,"""deep learning"" AND chemistry",1
3,"""machine learning"" AND materials",22
4,"""artificial intelligence"" AND materials",12
5,"""deep learning"" AND materials",6
6,"""machine learning"" AND catalysis",0
7,"""materials informatics""",0
8,cheminformatics,0
9,"""molecular machine learning""",0


In [17]:
NSF_QUERIES = [
    "machine AND learning",
    "artificial AND intelligence",
    "deep AND learning",
    "materials AND informatics",
    "cheminformatics",
    "molecular AND modeling",
    "molecular AND modelling",
    "reaction AND prediction",
    "computational AND chemistry",
    "data-driven AND chemistry",
    "data-driven AND materials",
    "autonomous AND laboratory",
    "self-driving AND laboratory"
]

In [18]:
query_test_results = []

for query in NSF_QUERIES:
    test_awards = fetch_nsf_awards_for_period(
        start_date=date(2021, 1, 1),
        end_date=date(2021, 1, 31),
        keyword=query
    )

    query_test_results.append({
        "query": query,
        "records_returned": len(test_awards)
    })

query_test_df = pd.DataFrame(query_test_results)

display(query_test_df)

2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 75
2021-01: page returned 25, total collected 100
2021-01: page returned 2, total collected 102
2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 8, total collected 58
2021-01: page returned 25, total collected 25
2021-01: page returned 19, total collected 44
2021-01: page returned 1, total collected 1
2021-01: page returned 0, total collected 0
2021-01: page returned 25, total collected 25
2021-01: page returned 3, total collected 28
2021-01: page returned 2, total collected 2
2021-01: page returned 0, total collected 0
2021-01: page returned 15, total collected 15
2021-01: page returned 25, total collected 25
2021-01: page returned 18, total collected 43
2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 

,query,records_returned
0,machine AND learning,102
1,artificial AND intelligence,58
2,deep AND learning,44
3,materials AND informatics,1
4,cheminformatics,0
5,molecular AND modeling,28
6,molecular AND modelling,2
7,reaction AND prediction,0
8,computational AND chemistry,15
9,data-driven AND chemistry,43


In [ ]:
test_params = {
    "dateStart": "01/01/2021",
    "dateEnd": "01/31/2021",
    "rpp": 25,
    "offset": 0
}

test_response = requests.get(
    NSF_API_URL,
    params=test_params,
    timeout=60
)

test_response.raise_for_status()
test_json = test_response.json()

print("Request URL:")
print(test_response.url)

print("\nMetadata:")
print(test_json["response"]["metadata"])

print(
    "\nRecords returned:",
    len(test_json["response"].get("award", []))
)

Request URL:
https://api.nsf.gov/services/v1/awards.json?dateStart=01%2F01%2F2021&dateEnd=01%2F31%2F2021&rpp=25&offset=0

Metadata:
{'offset': 0, 'rpp': 25, 'totalCount': 473}

Records returned: 25


In [40]:
test_awards = fetch_nsf_awards_for_period(
    start_date=date(2021, 1, 1),
    end_date=date(2021, 1, 31),
    keyword=NSF_QUERY
)

print(f"January 2021 awards retrieved: {len(test_awards):,}")

2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 75
2021-01: page returned 25, total collected 100
2021-01: page returned 25, total collected 125
2021-01: page returned 25, total collected 150
2021-01: page returned 25, total collected 175
2021-01: page returned 25, total collected 200
2021-01: page returned 25, total collected 225
2021-01: page returned 25, total collected 250
2021-01: page returned 25, total collected 275
2021-01: page returned 25, total collected 300
2021-01: page returned 25, total collected 325
2021-01: page returned 25, total collected 350
2021-01: page returned 25, total collected 375
2021-01: page returned 25, total collected 400
2021-01: page returned 25, total collected 425
2021-01: page returned 25, total collected 450
2021-01: page returned 25, total collected 475
2021-01: page returned 25, total collected 500
2021-01: page returned 25, total collected 525
2021-01: page re

In [ ]:
from calendar import monthrange
from datetime import date, datetime
from pathlib import Path
import time

import pandas as pd
import requests


NSF_API_URL = "https://api.nsf.gov/services/v1/awards.json"

 NSF_QUERIES = [
    '"machine learning" AND chemistry',
    '"artificial intelligence" AND chemistry',
    '"deep learning" AND chemistry',
    '"machine learning" AND materials',
    '"artificial intelligence" AND materials',
    '"deep learning" AND materials',
    '"machine learning" AND catalysis',
    '"materials informatics"',
    'cheminformatics',
    '"molecular machine learning"',
    '"reaction prediction"',
    '"autonomous laboratory"',
    '"self-driving laboratory"'
]


START_DATE = date(2021, 1, 1)
END_DATE = date(2021, 1, 31)

RAW_DATA_DIR = Path("../data/raw/nsf")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

In [51]:
def fetch_nsf_awards_for_period(
    start_date,
    end_date,
    keyword,
    delay=0.25
):
    """
    Retrieve all NSF awards matching one keyword phrase
    whose project start date falls within the specified period.
    """
    if not isinstance(keyword, str) or not keyword.strip():
        raise ValueError(
            "A non-empty keyword string is required. "
            "This prevents an unrestricted NSF download."
        )

    period_awards = []
    offset = 0
    results_per_page = 25

    while True:
        params = {
            "keyword": keyword,
            "startDateStart": start_date.strftime("%m/%d/%Y"),
            "startDateEnd": end_date.strftime("%m/%d/%Y"),
            "rpp": results_per_page,
            "offset": offset
        }

        response = requests.get(
            NSF_API_URL,
            params=params,
            timeout=60
        )

        response.raise_for_status()
        payload = response.json()

        response_data = payload.get("response", {})
        awards = response_data.get("award", []) or []

        if isinstance(awards, dict):
            awards = [awards]

        period_awards.extend(awards)

        if len(awards) < results_per_page:
            break

        offset += results_per_page

        if offset >= 3000:
            raise RuntimeError(
                f"Query reached the 3,000-record limit: {keyword}"
            )

        time.sleep(delay)

    print(
        f"{start_date:%Y-%m} | "
        f"{keyword}: {len(period_awards)} records"
    )

    return period_awards

Test Query

In [52]:
test_query = NSF_QUERIES[0]

test_awards = fetch_nsf_awards_for_period(
    start_date=START_DATE,
    end_date=END_DATE,
    keyword=test_query
)

print(f"Query tested: {test_query}")
print(f"Records retrieved: {len(test_awards):,}")

2021-01 | "machine learning chemistry": 0 records
Query tested: "machine learning chemistry"
Records retrieved: 0


In [53]:
from datetime import date, datetime
from pathlib import Path
import pandas as pd

START_DATE = date(2021, 1, 1)
END_DATE = date(2025, 12, 31)

RAW_DATA_DIR = Path("../data/raw/nsf")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

print("Date range:", START_DATE, "to", END_DATE)
print("Number of search queries:", len(NSF_QUERIES))
print("Output folder:", RAW_DATA_DIR.resolve())

Date range: 2021-01-01 to 2025-12-31
Number of search queries: 13
Output folder: C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf


In [19]:
all_nsf_awards = []

checkpoint_path = (
    RAW_DATA_DIR
    / f"nsf_grantscope_checkpoint_{EXTRACTION_DATE}.csv"
)

total_queries = len(NSF_QUERIES)

for query_number, query in enumerate(
    NSF_QUERIES,
    start=1
):
    print(
        f"\n{'=' * 70}\n"
        f"QUERY {query_number} OF {total_queries}: {query}\n"
        f"{'=' * 70}"
    )

    query_total = 0

    for window_start, window_end in generate_monthly_windows(
        START_DATE,
        END_DATE
    ):
        monthly_awards = fetch_nsf_awards_for_period(
            start_date=window_start,
            end_date=window_end,
            keyword=query
        )

        query_total += len(monthly_awards)

        for award in monthly_awards:
            award_copy = award.copy()

            # Record which search found the award.
            award_copy["_matched_query"] = query

            all_nsf_awards.append(award_copy)

    print(
        f"\nCompleted query {query_number} of "
        f"{total_queries}: {query}"
    )
    print(f"Records returned: {query_total:,}")

    # Save progress after each complete query.
    checkpoint_df = pd.DataFrame(all_nsf_awards)

    checkpoint_df.to_csv(
        checkpoint_path,
        index=False
    )

    print(
        f"Checkpoint saved: "
        f"{len(checkpoint_df):,} cumulative rows"
    )

print(
    "\nExtraction complete."
    f"\nRows before deduplication: "
    f"{len(all_nsf_awards):,}"
)


QUERY 1 OF 13: machine AND learning
2021-01: page returned 25, total collected 25
2021-01: page returned 25, total collected 50
2021-01: page returned 25, total collected 75
2021-01: page returned 25, total collected 100
2021-01: page returned 2, total collected 102
2021-02: page returned 25, total collected 25
2021-02: page returned 25, total collected 50
2021-02: page returned 18, total collected 68
2021-03: page returned 25, total collected 25
2021-03: page returned 25, total collected 50
2021-03: page returned 25, total collected 75
2021-03: page returned 11, total collected 86
2021-04: page returned 25, total collected 25
2021-04: page returned 25, total collected 50
2021-04: page returned 25, total collected 75
2021-04: page returned 3, total collected 78
2021-05: page returned 25, total collected 25
2021-05: page returned 25, total collected 50
2021-05: page returned 25, total collected 75
2021-05: page returned 25, total collected 100
2021-05: page returned 10, total collected

In [28]:

nsf_raw_df=pd.DataFrame(all_nsf_awards)

In [29]:
nsf_raw_df

,abstractText,activeAwd,agency,awardAgencyCode,awardee,awardeeAddress,awardeeCity,awardeeCountryCode,awardeeDistrict,awardeeDistrictCode,...,startDate,title,transType,ueiNumber,_matched_query,piMiddeInitial,coPDPI,awdSpAttnCode,awdSpAttnDesc,arraAmount
0,Composite structures have increasingly emerged...,false,NSF,4900,VIRGINIA POLYTECHNIC INSTITUTE & STATE UNIVERSITY,300 TURNER ST NW,BLACKSBURG,US,09,VA09,...,01/15/2021,Ultra-high Precision Assembly of Aerospace Com...,Standard Grant,QDE5UHE5XD16,machine AND learning,NaN,NaN,NaN,NaN,NaN
1,National efforts to digitize natural history c...,false,NSF,4900,UNIVERSITY OF FLORIDA,1523 UNION RD RM 207,GAINESVILLE,US,03,FL03,...,01/15/2021,Collaborative Research: CIBR: Leaping the Spec...,Standard Grant,NNFQH1JAPEP3,machine AND learning,P,NaN,NaN,NaN,NaN
2,In an effort to support decision making by gov...,false,NSF,4900,THE JOHNS HOPKINS UNIVERSITY,3400 N CHARLES ST,BALTIMORE,US,07,MD07,...,01/15/2021,RAPID: Real-time Forecasting of COVID-19 risk ...,Standard Grant,FTMTDMBR29C7,machine AND learning,M,NaN,NaN,NaN,NaN
3,COVID-19 disproportionately affects the low-wa...,false,NSF,4900,WAYNE STATE UNIVERSITY,5700 CASS AVE STE 4900,DETROIT,US,13,MI13,...,01/15/2021,SCC-CIVIC-PG Track A: Leveraging AI-assist Mic...,Standard Grant,M6K6NTJ2MNE5,machine AND learning,NaN,"[Daniel Grosu dgrosu@wayne.edu, Tierra Bills t...",NaN,NaN,NaN
4,The broader impact/commercial potential of thi...,false,NSF,4900,WAYNE STATE UNIVERSITY,5700 CASS AVE STE 4900,DETROIT,US,13,MI13,...,01/15/2021,I-Corps: AI-enabled automation intelligence s...,Standard Grant,M6K6NTJ2MNE5,machine AND learning,NaN,[Murat Yildirim murat@wayne.edu],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33120,Robotic systems that must operate in hazardous...,true,NSF,4900,THE UNIVERSITY CORPORATION,18111 NORDHOFF ST,NORTHRIDGE,US,32,CA32,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,LAGNHMC58DF3,self-driving AND laboratory,NaN,NaN,NaN,NaN,NaN
33121,Robotic systems that must operate in hazardous...,true,NSF,4900,UNIVERSITY OF WISCONSIN SYSTEM,21 N PARK ST STE 6301,MADISON,US,02,WI02,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,LCLSJAGTNZQ7,self-driving AND laboratory,NaN,"[Radu Serban serban@engr.wisc.edu, Luning Bakk...",NaN,NaN,NaN
33122,Robotic systems that must operate in hazardous...,true,NSF,4900,CAL POLY POMONA FOUNDATION INC,3801 W TEMPLE AVE,POMONA,US,35,CA35,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,JMGMMM7BMBT6,self-driving AND laboratory,NaN,NaN,NaN,NaN,NaN
33123,Robotic systems that must operate in hazardous...,true,NSF,4900,CSU FULLERTON AUXILIARY SERVICES CORPORATION,1121 N STATE COLLEGE BLVD,FULLERTON,US,45,CA45,...,11/01/2025,Collaborative Research: CyberTraining: Impleme...,Standard Grant,VQ5WK498QDC6,self-driving AND laboratory,NaN,NaN,NaN,NaN,NaN


In [23]:
Query_df.columns


Index(['abstractText', 'activeAwd', 'agency', 'awardAgencyCode', 'awardee',
       'awardeeAddress', 'awardeeCity', 'awardeeCountryCode',
       'awardeeDistrict', 'awardeeDistrictCode', 'awardeeName', 'awardeePhone',
       'awardeeStateCode', 'awardeeZipCode', 'cfdaNumber', 'date', 'dirAbbr',
       'divAbbr', 'estimatedTotalAmt', 'expDate', 'fundAgencyCode',
       'fundProgramName', 'fundsObligated', 'fundsObligatedAmt', 'histAwd',
       'id', 'initAmendmentDate', 'jrnl', 'latestAmendmentDate', 'managingPec',
       'orgCodeDir', 'orgCodeDiv', 'orgLongName', 'orgLongName2', 'orgUrl',
       'parentUeiNumber', 'pdPIName', 'perfAddress', 'perfCity',
       'perfCountryCode', 'perfDistrict', 'perfDistrictCode', 'perfLocation',
       'perfStateCode', 'perfZipCode', 'pi', 'piEmail', 'piFirstName', 'piId',
       'piLastName', 'poEmail', 'poName', 'poPhone', 'primaryProgram',
       'progEleCode', 'program', 'progRefCode', 'projectOutComesReport',
       'publicAccessMandate', 'publica

In [2]:
from pathlib import Path

RAW_DATA_DIR = Path("../Data/Raw_Data")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Saving files to:")
print(RAW_DATA_DIR.resolve())

Saving files to:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data


In [7]:
from pathlib import Path
from datetime import datetime

print("Current notebook location:")
print(Path.cwd())

SEARCH_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
)

data_extensions = {".csv", ".jsonl", ".parquet"}

found_files = []

for file in SEARCH_ROOT.rglob("*"):
    if not file.is_file():
        continue

    filename = file.name.lower()

    if (
        file.suffix.lower() in data_extensions
        and (
            "nsf" in filename
            or "grantscope" in filename
            or "checkpoint" in filename
        )
    ):
        found_files.append(file)

found_files = sorted(
    set(found_files),
    key=lambda file: file.stat().st_size,
    reverse=True
)

print(f"\nPotential NSF files found: {len(found_files)}\n")

for file in found_files:
    size_mb = file.stat().st_size / 1_000_000
    modified = datetime.fromtimestamp(
        file.stat().st_mtime
    )

    print(f"{size_mb:,.2f} MB | {modified}")
    print(file)
    print()

Current notebook location:
c:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Notebooks

Potential NSF files found: 2

300.18 MB | 2026-08-01 13:12:32.377770
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf\nsf_grantscope_raw_2021_2025_2026-08-01.csv

300.18 MB | 2026-08-01 12:55:07.874020
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf\nsf_grantscope_checkpoint_2026-08-01.csv



In [8]:
import pandas as pd

csv_files = [
    file
    for file in found_files
    if file.suffix.lower() == ".csv"
]

if not csv_files:
    raise FileNotFoundError(
        "No matching CSV was found."
    )

largest_csv = max(
    csv_files,
    key=lambda file: file.stat().st_size
)

print("Loading:")
print(largest_csv)

nsf_raw_df = pd.read_csv(
    largest_csv,
    low_memory=False
)

print(
    f"Loaded: {nsf_raw_df.shape[0]:,} rows × "
    f"{nsf_raw_df.shape[1]:,} columns"
)

Loading:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf\nsf_grantscope_raw_2021_2025_2026-08-01.csv
Loaded: 33,125 rows × 70 columns


In [9]:
from pathlib import Path
import shutil

current_file = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\data\raw\nsf"
    r"\nsf_grantscope_raw_2021_2025_2026-08-01.csv"
)

target_folder = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project"
    r"\GrantScopeAI\Data\Raw_Data"
)

target_folder.mkdir(parents=True, exist_ok=True)

target_file = target_folder / current_file.name

shutil.move(current_file, target_file)

print("Moved to:")
print(target_file)
print("File exists:", target_file.exists())

Moved to:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\nsf_grantscope_raw_2021_2025_2026-08-01.csv
File exists: True


In [10]:
import pandas as pd

nsf_raw_df = pd.read_csv(
    target_file,
    low_memory=False
)

print(nsf_raw_df.shape)

(33125, 70)


## OpenAlex Source Validation and Publication-Momentum Acquisition

OpenAlex provides publication-level metadata for assessing research
activity around AI-enabled chemistry and materials topics.

This section validates API access and creates a topic-by-year summary
for the 2021–2025 project period.

In [12]:
from getpass import getpass

OPENALEX_API_KEY = getpass(
    "Paste your OpenAlex API key: "
)

if not OPENALEX_API_KEY:
    raise ValueError("An OpenAlex API key is required.")

print("OpenAlex API key loaded.")

OpenAlex API key loaded.


In [13]:
import requests

test_response = requests.get(
    "https://api.openalex.org/works",
    params={
        "api_key": OPENALEX_API_KEY,
        "search": "machine learning chemistry",
        "per_page": 1
    },
    timeout=60
)

print("Status code:", test_response.status_code)

test_response.raise_for_status()

test_data = test_response.json()

print(
    "Matching works:",
    f"{test_data['meta']['count']:,}"
)

Status code: 200
Matching works: 279,510


In [14]:
test_query = (
    '("machine learning" OR "artificial intelligence" '
    'OR "deep learning") '
    'AND (chemistry OR chemical)'
)

test_response = requests.get(
    "https://api.openalex.org/works",
    params={
        "api_key": OPENALEX_API_KEY,
        "search": test_query,
        "filter": "publication_year:2021",
        "per_page": 1,
        "select": "id,title,publication_year"
    },
    timeout=60
)

test_response.raise_for_status()
test_data = test_response.json()

print("Status code:", test_response.status_code)
print(
    "Matching 2021 publications:",
    f"{test_data['meta']['count']:,}"
)

Status code: 200
Matching 2021 publications: 40,651


In [15]:
OPENALEX_QUERIES = {
    "AI-enabled chemistry": (
        '("machine learning" OR "artificial intelligence" '
        'OR "deep learning") '
        'AND (chemistry OR chemical)'
    ),

    "AI-enabled materials": (
        '("machine learning" OR "artificial intelligence" '
        'OR "deep learning") '
        'AND ("materials science" OR materials)'
    ),

    "Materials informatics": '"materials informatics"',

    "Cheminformatics": "cheminformatics",

    "Molecular machine learning": (
        '"molecular machine learning"'
    ),

    "Reaction prediction": (
        '"reaction prediction" '
        'AND ("machine learning" OR "artificial intelligence")'
    ),

    "AI-enabled catalysis": (
        '("machine learning" OR "artificial intelligence") '
        'AND (catalysis OR catalyst)'
    ),

    "Autonomous laboratories": (
        '"autonomous laboratory" '
        'OR "self-driving laboratory"'
    )
}

OPENALEX_YEARS = range(2021, 2026)

In [16]:
import time
import pandas as pd

openalex_summary_rows = []

for topic_name, search_query in OPENALEX_QUERIES.items():
    print(f"\nTopic: {topic_name}")

    for year in OPENALEX_YEARS:
        response = requests.get(
            "https://api.openalex.org/works",
            params={
                "api_key": OPENALEX_API_KEY,
                "search": search_query,
                "filter": f"publication_year:{year}",
                "per_page": 1,
                "select": "id"
            },
            timeout=60
        )

        response.raise_for_status()
        payload = response.json()

        publication_count = int(
            payload.get("meta", {}).get("count", 0) or 0
        )

        openalex_summary_rows.append({
            "topic": topic_name,
            "publication_year": year,
            "publication_count": publication_count,
            "search_query": search_query,
            "source": "OpenAlex"
        })

        print(f"{year}: {publication_count:,}")

        time.sleep(0.1)

openalex_topic_year_df = pd.DataFrame(openalex_summary_rows)

print("\nFinal shape:", openalex_topic_year_df.shape)
display(openalex_topic_year_df)


Topic: AI-enabled chemistry
2021: 40,651
2022: 54,337
2023: 96,605
2024: 112,587
2025: 142,246

Topic: AI-enabled materials
2021: 107,790
2022: 146,376
2023: 250,712
2024: 295,260
2025: 386,893

Topic: Materials informatics
2021: 398
2022: 511
2023: 807
2024: 836
2025: 1,021

Topic: Cheminformatics
2021: 2,511
2022: 2,997
2023: 4,606
2024: 4,590
2025: 4,975

Topic: Molecular machine learning
2021: 189
2022: 263
2023: 633
2024: 633
2025: 549

Topic: Reaction prediction
2021: 197
2022: 256
2023: 484
2024: 505
2025: 576

Topic: AI-enabled catalysis
2021: 6,094
2022: 8,449
2023: 18,137
2024: 25,744
2025: 39,893

Topic: Autonomous laboratories
2021: 75
2022: 119
2023: 273
2024: 440
2025: 859

Final shape: (40, 5)


,topic,publication_year,publication_count,search_query,source
0,AI-enabled chemistry,2021,40651,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
1,AI-enabled chemistry,2022,54337,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
2,AI-enabled chemistry,2023,96605,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
3,AI-enabled chemistry,2024,112587,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
4,AI-enabled chemistry,2025,142246,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
5,AI-enabled materials,2021,107790,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
6,AI-enabled materials,2022,146376,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
7,AI-enabled materials,2023,250712,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
8,AI-enabled materials,2024,295260,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex
9,AI-enabled materials,2025,386893,"(""machine learning"" OR ""artificial intelligenc...",OpenAlex


In [17]:
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI"
)

RAW_DATA_DIR = PROJECT_ROOT / "Data" / "Raw_Data"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

openalex_csv_path = (
    RAW_DATA_DIR
    / f"openalex_topic_year_counts_2021_2025_{EXTRACTION_DATE}.csv"
)

openalex_topic_year_df.to_csv(
    openalex_csv_path,
    index=False
)

print("Saved successfully:")
print(openalex_csv_path)
print("File exists:", openalex_csv_path.exists())

Saved successfully:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\openalex_topic_year_counts_2021_2025_2026-08-01.csv
File exists: True


## CORDIS EURIO Source Validation

CORDIS provides information on European Union-funded research and
innovation projects.

This section validates access to the public EURIO SPARQL endpoint and
tests whether it provides project identifiers, titles, dates, scientific
classifications, and other metadata required for GrantScopeAI.

In [21]:
CORDIS_SPARQL_URL = "https://cordis.europa.eu/datalab/sparql"

In [22]:
cordis_test_query = """
PREFIX eurio: <http://data.europa.eu/s66#>

SELECT ?project ?project_title
WHERE {
    ?project a eurio:Project .
    ?project eurio:title ?project_title .
}
LIMIT 10
"""

In [23]:
cordis_response = requests.post(
    CORDIS_SPARQL_URL,
    data={
        "query": cordis_test_query
    },
    headers={
        "Accept": "application/sparql-results+json"
    },
    timeout=90
)

print("Status code:", cordis_response.status_code)
print(
    "Content type:",
    cordis_response.headers.get("content-type")
)

if cordis_response.status_code != 200:
    print(cordis_response.text[:1000])

cordis_response.raise_for_status()

Status code: 200
Content type: application/sparql-results+json


In [18]:
import pandas as pd
import requests

CORDIS_SPARQL_URL = (
    "https://cordis.europa.eu/datalab/sparql-endpoint"
)

cordis_test_query = """
PREFIX eurio: <http://data.europa.eu/s66#>
PREFIX skos-xl: <http://www.w3.org/2008/05/skos-xl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT DISTINCT
    ?project
    ?project_id
    ?project_title
    ?start_date
    ?end_date
WHERE {
    ?project a eurio:Project ;
             eurio:identifier ?project_id ;
             eurio:title ?project_title ;
             eurio:startDate ?start_date ;
             eurio:hasEuroSciVocClassification ?classification .

    OPTIONAL {
        ?project eurio:endDate ?end_date .
    }

    ?classification skos-xl:prefLabel ?label .
    ?label skos-xl:literalForm "artificial intelligence"@en .

    FILTER(
        YEAR(?start_date) >= 2021 &&
        YEAR(?start_date) <= 2025
    )
}
ORDER BY DESC(?start_date)
LIMIT 25
"""

In [24]:
cordis_response = requests.get(
    CORDIS_SPARQL_URL,
    params={
        "query": cordis_test_query,
        "format": "application/sparql-results+json"
    },
    headers={
        "Accept": "application/sparql-results+json"
    },
    timeout=90
)

print("Status code:", cordis_response.status_code)
print("Content type:", cordis_response.headers.get("content-type"))

cordis_response.raise_for_status()

Status code: 200
Content type: application/sparql-results+json


### CORDIS Access Validation Result

The public CORDIS EURIO SPARQL endpoint was successfully accessed.

The validation queries returned project identifiers, titles, start dates,
end dates, and EuroSciVoc artificial-intelligence classifications.

For the full project acquisition, the official CORDIS bulk CSV datasets
will be used because they provide complete project, funding, programme,
organisation, and country metadata in a tabular format. The SPARQL
endpoint remains available for targeted validation and supplementary
queries.

In [25]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI"
)

CORDIS_RAW_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Raw_Data"
    / "CORDIS"
)

HORIZON_EUROPE_DIR = CORDIS_RAW_DIR / "Horizon_Europe"
HORIZON_2020_DIR = CORDIS_RAW_DIR / "Horizon_2020"

HORIZON_EUROPE_DIR.mkdir(parents=True, exist_ok=True)
HORIZON_2020_DIR.mkdir(parents=True, exist_ok=True)

print(HORIZON_EUROPE_DIR)
print(HORIZON_2020_DIR)

C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\CORDIS\Horizon_Europe
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\CORDIS\Horizon_2020


In [32]:
cordis_files = sorted(
    CORDIS_RAW_DIR.rglob("*")
)

for file in cordis_files:
    if file.is_file():
        size_mb = file.stat().st_size / 1_000_000

        print(
            f"{size_mb:,.2f} MB | "
            f"{file.relative_to(CORDIS_RAW_DIR)}"
        )

In [27]:
cordis_csv_files = sorted(
    CORDIS_RAW_DIR.rglob("*.csv")
)

print(f"CSV files found: {len(cordis_csv_files)}\n")

for file in cordis_csv_files:
    print(file.relative_to(CORDIS_RAW_DIR))

CSV files found: 0



In [28]:
import pandas as pd


def inspect_cordis_csv(file_path, rows=5):
    """
    Load a small sample from a CORDIS CSV file.
    """
    try:
        sample_df = pd.read_csv(
            file_path,
            sep=";",
            nrows=rows,
            low_memory=False
        )
    except UnicodeDecodeError:
        sample_df = pd.read_csv(
            file_path,
            sep=";",
            encoding="latin-1",
            nrows=rows,
            low_memory=False
        )

    print(f"\nFile: {file_path.name}")
    print(f"Sample shape: {sample_df.shape}")
    print("Columns:")
    print(sample_df.columns.tolist())

    return sample_df

In [29]:
project_file_candidates = [
    file
    for file in cordis_csv_files
    if "project" in file.name.lower()
]

for file in project_file_candidates:
    inspect_cordis_csv(file)

In [30]:
print("CORDIS folder being searched:")
print(CORDIS_RAW_DIR.resolve())
print("Folder exists:", CORDIS_RAW_DIR.exists())

CORDIS folder being searched:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\CORDIS
Folder exists: True


In [31]:
all_cordis_files = sorted(
    file
    for file in CORDIS_RAW_DIR.rglob("*")
    if file.is_file()
)

print(f"Files found: {len(all_cordis_files)}\n")

for file in all_cordis_files:
    size_mb = file.stat().st_size / 1_000_000

    print(
        f"{size_mb:,.2f} MB | "
        f"{file.relative_to(CORDIS_RAW_DIR)}"
    )

Files found: 0



# Cordis csv Inspection

In [9]:
from pathlib import Path
import pandas as pd

HORIZON_EUROPE_DIR = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI\Data\Raw_Data"
    r"\CORDIS\Horizon_Europe"
)

def load_standard_cordis_csv(filename):
    """
    Load a standard semicolon-delimited CORDIS CSV.
    """
    return pd.read_csv(
        HORIZON_EUROPE_DIR / filename,
        sep=";",
        encoding="utf-8-sig",
        low_memory=False
    )


cordis_organizations_df = load_standard_cordis_csv(
    "organization.csv"
)

cordis_scivoc_df = load_standard_cordis_csv(
    "euroSciVoc.csv"
)

cordis_topics_df = load_standard_cordis_csv(
    "topics.csv"
)

cordis_legal_df = load_standard_cordis_csv(
    "legalBasis.csv"
)

cordis_links_df = load_standard_cordis_csv(
    "webLink.csv"
)

In [10]:
for name, dataframe in {
    "organizations": cordis_organizations_df,
    "euroSciVoc": cordis_scivoc_df,
    "topics": cordis_topics_df,
    "legalBasis": cordis_legal_df,
    "webLinks": cordis_links_df
}.items():
    print(
        f"{name}: "
        f"{dataframe.shape[0]:,} rows × "
        f"{dataframe.shape[1]} columns"
    )

organizations: 144,117 rows × 25 columns
euroSciVoc: 51,718 rows × 5 columns
topics: 23,278 rows × 3 columns
legalBasis: 30,044 rows × 4 columns
webLinks: 57,859 rows × 9 columns


In [11]:
import csv

EXPECTED_PROJECT_COLUMNS = 22
OBJECTIVE_INDEX = 15
TRAILING_COLUMNS_AFTER_OBJECTIVE = 6

unrepaired_project_rows = []


def repair_project_row(fields):
    """
    Repair rows containing extra semicolons inside the
    CORDIS project objective field.
    """
    if len(fields) == EXPECTED_PROJECT_COLUMNS:
        return fields

    if len(fields) > EXPECTED_PROJECT_COLUMNS:
        objective_end = (
            len(fields)
            - TRAILING_COLUMNS_AFTER_OBJECTIVE
        )

        repaired_fields = (
            fields[:OBJECTIVE_INDEX]
            + [
                ";".join(
                    fields[
                        OBJECTIVE_INDEX:objective_end
                    ]
                )
            ]
            + fields[objective_end:]
        )

        if len(repaired_fields) == EXPECTED_PROJECT_COLUMNS:
            return repaired_fields

    unrepaired_project_rows.append(fields)
    return None

In [12]:
cordis_projects_df = pd.read_csv(
    HORIZON_EUROPE_DIR / "project.csv",
    sep=";",
    encoding="utf-8-sig",
    engine="python",
    quoting=csv.QUOTE_NONE,
    on_bad_lines=repair_project_row,
    dtype=str
)

print(
    f"Projects loaded: "
    f"{cordis_projects_df.shape[0]:,} rows × "
    f"{cordis_projects_df.shape[1]} columns"
)

print(
    "Rows that could not be repaired:",
    len(unrepaired_project_rows)
)

Projects loaded: 23,278 rows × 22 columns
Rows that could not be repaired: 0


In [14]:
print(cordis_projects_df.columns.tolist())

['"id"', '"acronym"', '"status"', '"title"', '"startDate"', '"endDate"', '"totalCost"', '"ecMaxContribution"', '"topics"', '"ecSignatureDate"', '"frameworkProgramme"', '"masterCall"', '"subCall"', '"fundingScheme"', '"nature"', '"objective"', '"contentUpdateDate"', '"rcn"', '"grantDoi"', '"keywords"', '"Human-validated"', '"legalBasis"']


In [15]:
cordis_projects_df.columns = (
    cordis_projects_df.columns
    .str.strip()
    .str.replace('"', '', regex=False)
    .str.replace('\ufeff', '', regex=False)
)

print(cordis_projects_df.columns.tolist())

['id', 'acronym', 'status', 'title', 'startDate', 'endDate', 'totalCost', 'ecMaxContribution', 'topics', 'ecSignatureDate', 'frameworkProgramme', 'masterCall', 'subCall', 'fundingScheme', 'nature', 'objective', 'contentUpdateDate', 'rcn', 'grantDoi', 'keywords', 'Human-validated', 'legalBasis']


In [16]:
print("Columns:", len(cordis_projects_df.columns))
print("Rows:", len(cordis_projects_df))

print(
    "Unique project IDs:",
    cordis_projects_df["id"].nunique()
)

print(
    "Missing project IDs:",
    cordis_projects_df["id"].isna().sum()
)

print(
    "Duplicate project IDs:",
    cordis_projects_df["id"].duplicated().sum()
)

Columns: 22
Rows: 23278
Unique project IDs: 23278
Missing project IDs: 0
Duplicate project IDs: 0


In [17]:
required_columns = [
    "id",
    "title",
    "objective",
    "startDate",
    "endDate",
    "totalCost",
    "ecMaxContribution"
]

missing_columns = [
    column
    for column in required_columns
    if column not in cordis_projects_df.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [18]:
cordis_dataframes = {
    "projects": cordis_projects_df,
    "organizations": cordis_organizations_df,
    "euroSciVoc": cordis_scivoc_df,
    "topics": cordis_topics_df,
    "legalBasis": cordis_legal_df,
    "webLinks": cordis_links_df
}

for name, dataframe in cordis_dataframes.items():
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.replace('"', "", regex=False)
        .str.replace("\ufeff", "", regex=False)
    )

    print(f"{name}: {len(dataframe.columns)} columns")

projects: 22 columns
organizations: 25 columns
euroSciVoc: 5 columns
topics: 3 columns
legalBasis: 4 columns
webLinks: 9 columns


In [19]:
join_key_checks = {
    "projects": "id",
    "organizations": "projectID",
    "euroSciVoc": "projectID",
    "topics": "projectID",
    "legalBasis": "projectID",
    "webLinks": "projectID"
}

for table_name, key_column in join_key_checks.items():
    dataframe = cordis_dataframes[table_name]

    print(
        f"{table_name}: "
        f"'{key_column}' present = "
        f"{key_column in dataframe.columns}"
    )

projects: 'id' present = True
organizations: 'projectID' present = True
euroSciVoc: 'projectID' present = True
topics: 'projectID' present = True
legalBasis: 'projectID' present = True
webLinks: 'projectID' present = True


In [20]:
cordis_table_summary_rows = []

for table_name, key_column in join_key_checks.items():
    dataframe = cordis_dataframes[table_name]

    cordis_table_summary_rows.append({
        "table": table_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "join_key": key_column,
        "missing_join_keys": dataframe[key_column].isna().sum(),
        "unique_projects": dataframe[key_column].nunique(),
        "duplicate_key_rows": dataframe[key_column].duplicated().sum()
    })

cordis_table_summary = pd.DataFrame(
    cordis_table_summary_rows
)

display(cordis_table_summary)

,table,rows,columns,join_key,missing_join_keys,unique_projects,duplicate_key_rows
0,projects,23278,22,id,0,23278,0
1,organizations,144117,25,projectID,0,23278,120839
2,euroSciVoc,51718,5,projectID,0,20062,31656
3,topics,23278,3,projectID,0,23278,0
4,legalBasis,30044,4,projectID,0,23278,6766
5,webLinks,57859,9,projectID,0,11081,46778


In [21]:
required_project_columns = [
    "id",
    "acronym",
    "title",
    "objective",
    "startDate",
    "endDate",
    "totalCost",
    "ecMaxContribution",
    "frameworkProgramme",
    "fundingScheme"
]

missing_project_columns = [
    column
    for column in required_project_columns
    if column not in cordis_projects_df.columns
]

print("Missing required columns:", missing_project_columns)
print("Project rows:", f"{len(cordis_projects_df):,}")
print(
    "Unique project IDs:",
    f"{cordis_projects_df['id'].nunique():,}"
)
print(
    "Duplicate project IDs:",
    cordis_projects_df["id"].duplicated().sum()
)

Missing required columns: []
Project rows: 23,278
Unique project IDs: 23,278
Duplicate project IDs: 0


In [22]:
cordis_date_columns = [
    "startDate",
    "endDate",
    "ecSignatureDate",
    "contentUpdateDate"
]

for column in cordis_date_columns:
    if column in cordis_projects_df.columns:
        cordis_projects_df[column] = pd.to_datetime(
            cordis_projects_df[column],
            errors="coerce"
        )

C:\Users\kahau\AppData\Local\Temp\ipykernel_3692\1259286013.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cordis_projects_df[column] = pd.to_datetime(
C:\Users\kahau\AppData\Local\Temp\ipykernel_3692\1259286013.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cordis_projects_df[column] = pd.to_datetime(
C:\Users\kahau\AppData\Local\Temp\ipykernel_3692\1259286013.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cordis_projects_df[column] = pd.to_datetime(
C:\Users\kahau\AppData\Local\Temp\ipykernel_3692\1259286013.py:10: UserWarning: Could not infer format, so

In [23]:
print(
    "Earliest project start:",
    cordis_projects_df["startDate"].min()
)

print(
    "Latest project start:",
    cordis_projects_df["startDate"].max()
)

display(
    cordis_projects_df["startDate"]
    .dt.year
    .value_counts(dropna=False)
    .sort_index()
)

Earliest project start: NaT
Latest project start: NaT


startDate
NaN    23278
Name: count, dtype: int64

In [24]:
cordis_scope_df = (
    cordis_projects_df[
        cordis_projects_df["startDate"]
        .dt.year
        .between(2021, 2025)
    ]
    .copy()
)

print(
    "All Horizon Europe projects:",
    f"{len(cordis_projects_df):,}"
)

print(
    "Projects starting during 2021–2025:",
    f"{len(cordis_scope_df):,}"
)

All Horizon Europe projects: 23,278
Projects starting during 2021–2025: 0


In [25]:
text_quality_summary = pd.DataFrame({
    "field": [
        "title",
        "objective",
        "keywords",
        "topics"
    ],
    "missing_count": [
        cordis_projects_df[column].isna().sum()
        for column in [
            "title",
            "objective",
            "keywords",
            "topics"
        ]
    ],
    "missing_percent": [
        round(
            cordis_projects_df[column]
            .isna()
            .mean()
            * 100,
            2
        )
        for column in [
            "title",
            "objective",
            "keywords",
            "topics"
        ]
    ]
})

display(text_quality_summary)

,field,missing_count,missing_percent
0,title,0,0.0
1,objective,0,0.0
2,keywords,0,0.0
3,topics,0,0.0


In [26]:
project_ids = set(
    cordis_projects_df["id"]
    .dropna()
    .astype(str)
)

coverage_rows = []

for table_name in [
    "organizations",
    "euroSciVoc",
    "topics",
    "legalBasis",
    "webLinks"
]:
    dataframe = cordis_dataframes[table_name]

    secondary_ids = set(
        dataframe["projectID"]
        .dropna()
        .astype(str)
    )

    matched_projects = len(
        project_ids.intersection(secondary_ids)
    )

    coverage_rows.append({
        "table": table_name,
        "projects_with_records": matched_projects,
        "total_projects": len(project_ids),
        "coverage_percent": round(
            matched_projects
            / len(project_ids)
            * 100,
            2
        )
    })

cordis_coverage_summary = pd.DataFrame(
    coverage_rows
)

display(cordis_coverage_summary)

,table,projects_with_records,total_projects,coverage_percent
0,organizations,0,23278,0.0
1,euroSciVoc,0,23278,0.0
2,topics,0,23278,0.0
3,legalBasis,0,23278,0.0
4,webLinks,0,23278,0.0


In [28]:
id_sources = {
    "projects": cordis_projects_df["id"],
    "organizations": cordis_organizations_df["projectID"],
    "euroSciVoc": cordis_scivoc_df["projectID"],
    "topics": cordis_topics_df["projectID"],
    "legalBasis": cordis_legal_df["projectID"],
    "webLinks": cordis_links_df["projectID"]
}

for name, series in id_sources.items():
    print(f"\n{name}")
    print("Data type:", series.dtype)
    print("Examples:")

    for value in series.dropna().head(5):
        print(repr(value))


projects
Data type: object
Examples:
'"101069359"'
'"101069357"'
'"101069586"'
'"101069604"'
'"101069529"'

organizations
Data type: int64
Examples:
101069359
101069359
101069359
101069359
101069359

euroSciVoc
Data type: int64
Examples:
101069359
101069359
101069359
101069359
101069357

topics
Data type: int64
Examples:
101069359
101069357
101069586
101069604
101069529

legalBasis
Data type: int64
Examples:
101069359
101069357
101069357
101069586
101069586

webLinks
Data type: int64
Examples:
101069359
101069359
101069359
101069359
101069359


In [29]:
def clean_cordis_project_id(series):
    """
    Convert CORDIS project identifiers to a consistent string format.
    """
    return (
        series
        .astype("string")
        .str.strip()
        .str.strip('"')
        .str.strip("'")
        .str.replace(r"\.0$", "", regex=True)
    )

In [30]:
cordis_projects_df["project_id_clean"] = clean_cordis_project_id(
    cordis_projects_df["id"]
)

cordis_organizations_df["project_id_clean"] = clean_cordis_project_id(
    cordis_organizations_df["projectID"]
)

cordis_scivoc_df["project_id_clean"] = clean_cordis_project_id(
    cordis_scivoc_df["projectID"]
)

cordis_topics_df["project_id_clean"] = clean_cordis_project_id(
    cordis_topics_df["projectID"]
)

cordis_legal_df["project_id_clean"] = clean_cordis_project_id(
    cordis_legal_df["projectID"]
)

cordis_links_df["project_id_clean"] = clean_cordis_project_id(
    cordis_links_df["projectID"]
)

In [31]:
display(
    cordis_projects_df[
        ["id", "project_id_clean"]
    ].head()
)

display(
    cordis_organizations_df[
        ["projectID", "project_id_clean"]
    ].head()
)

,id,project_id_clean
0,"""101069359""",101069359
1,"""101069357""",101069357
2,"""101069586""",101069586
3,"""101069604""",101069604
4,"""101069529""",101069529


,projectID,project_id_clean
0,101069359,101069359
1,101069359,101069359
2,101069359,101069359
3,101069359,101069359
4,101069359,101069359


In [32]:
project_ids = set(
    cordis_projects_df["project_id_clean"].dropna()
)

organization_ids = set(
    cordis_organizations_df["project_id_clean"].dropna()
)

print(
    "Matching project IDs:",
    len(project_ids.intersection(organization_ids))
)

Matching project IDs: 23278


In [33]:
secondary_tables = {
    "organizations": cordis_organizations_df,
    "euroSciVoc": cordis_scivoc_df,
    "topics": cordis_topics_df,
    "legalBasis": cordis_legal_df,
    "webLinks": cordis_links_df
}

coverage_rows = []

for table_name, dataframe in secondary_tables.items():
    secondary_ids = set(
        dataframe["project_id_clean"].dropna()
    )

    matching_ids = project_ids.intersection(
        secondary_ids
    )

    coverage_rows.append({
        "table": table_name,
        "projects_with_records": len(matching_ids),
        "total_projects": len(project_ids),
        "coverage_percent": round(
            len(matching_ids) / len(project_ids) * 100,
            2
        )
    })

cordis_coverage_summary = pd.DataFrame(
    coverage_rows
)

display(cordis_coverage_summary)

,table,projects_with_records,total_projects,coverage_percent
0,organizations,23278,23278,100.00
1,euroSciVoc,20062,23278,86.18
2,topics,23278,23278,100.00
3,legalBasis,23278,23278,100.00
4,webLinks,11081,23278,47.60


In [36]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Processed_Data"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

coverage_output_path = (
    PROCESSED_DATA_DIR
    / "cordis_source_coverage_summary.csv"
)

cordis_coverage_summary.to_csv(
    coverage_output_path,
    index=False
)

print("Coverage summary saved successfully.")
print("File location:")
print(coverage_output_path.resolve())
print("File exists:", coverage_output_path.exists())


Coverage summary saved successfully.
File location:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\cordis_source_coverage_summary.csv
File exists: True


## Source Comparison

The following table compares the three primary data sources selected for
GrantScopeAI, their analytical roles, acquisition methods, current status,
and known limitations.

In [37]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI"
)

REFERENCE_DATA_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Reference_Data"
)

REFERENCE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

source_comparison_df = pd.DataFrame([
    {
        "source": "NSF Award Search",
        "region": "United States",
        "analytical_role": "US research grants and funded projects",
        "access_method": "NSF Award Search REST API",
        "authentication": "None",
        "time_scope": "2021–2025 project start dates",
        "data_granularity": "One row per API result before deduplication",
        "current_dataset": "33,125 candidate rows and 70 columns",
        "important_text_fields": "title; abstractText",
        "funding_fields": (
            "estimatedTotalAmt; fundsObligatedAmt"
        ),
        "organisation_fields": (
            "awardeeName; awardeeCountryCode; "
            "dirAbbr; divAbbr"
        ),
        "current_status": (
            "Acquired and saved locally; "
            "deduplication and relevance filtering pending"
        ),
        "main_limitations": (
            "Broad keyword searches create false positives; "
            "candidate rows include duplicate awards"
        )
    },
    {
        "source": "OpenAlex",
        "region": "International",
        "analytical_role": (
            "Publication activity and research-momentum context"
        ),
        "access_method": "OpenAlex Works REST API",
        "authentication": "Free API key",
        "time_scope": "2021–2025 publication years",
        "data_granularity": "One row per topic-year combination",
        "current_dataset": "40 topic-year summary rows",
        "important_text_fields": "topic; search_query",
        "funding_fields": "Not used in initial summary",
        "organisation_fields": (
            "Available through publication-level records, "
            "but not yet extracted"
        ),
        "current_status": (
            "Topic-year publication counts acquired and saved"
        ),
        "main_limitations": (
            "Topic categories overlap; current dataset contains "
            "counts rather than publication-level records"
        )
    },
    {
        "source": "CORDIS",
        "region": "European Union",
        "analytical_role": (
            "EU-funded research projects and participants"
        ),
        "access_method": (
            "EURIO SPARQL validation and Horizon Europe bulk CSV"
        ),
        "authentication": "None for SPARQL and bulk downloads",
        "time_scope": "Horizon Europe projects, initially 2021–2025",
        "data_granularity": (
            "One central project table plus related supporting tables"
        ),
        "current_dataset": "23,278 Horizon Europe projects",
        "important_text_fields": (
            "title; objective; keywords; EuroSciVoc classifications"
        ),
        "funding_fields": (
            "totalCost; ecMaxContribution; ecContribution"
        ),
        "organisation_fields": (
            "name; country; role; activityType"
        ),
        "current_status": (
            "SPARQL validated and six Horizon Europe tables loaded"
        ),
        "main_limitations": (
            "Supporting tables are one-to-many; project.csv required "
            "quote and delimiter repair; Horizon 2020 not yet integrated"
        )
    }
])

display(source_comparison_df)

,source,region,analytical_role,access_method,authentication,time_scope,data_granularity,current_dataset,important_text_fields,funding_fields,organisation_fields,current_status,main_limitations
0,NSF Award Search,United States,US research grants and funded projects,NSF Award Search REST API,None,2021–2025 project start dates,One row per API result before deduplication,"33,125 candidate rows and 70 columns",title; abstractText,estimatedTotalAmt; fundsObligatedAmt,awardeeName; awardeeCountryCode; dirAbbr; divAbbr,Acquired and saved locally; deduplication and ...,Broad keyword searches create false positives;...
1,OpenAlex,International,Publication activity and research-momentum con...,OpenAlex Works REST API,Free API key,2021–2025 publication years,One row per topic-year combination,40 topic-year summary rows,topic; search_query,Not used in initial summary,"Available through publication-level records, b...",Topic-year publication counts acquired and saved,Topic categories overlap; current dataset cont...
2,CORDIS,European Union,EU-funded research projects and participants,EURIO SPARQL validation and Horizon Europe bul...,None for SPARQL and bulk downloads,"Horizon Europe projects, initially 2021–2025",One central project table plus related support...,"23,278 Horizon Europe projects",title; objective; keywords; EuroSciVoc classif...,totalCost; ecMaxContribution; ecContribution,name; country; role; activityType,SPARQL validated and six Horizon Europe tables...,Supporting tables are one-to-many; project.csv...


In [38]:
source_comparison_path = (
    REFERENCE_DATA_DIR
    / "source_comparison_table.csv"
)

source_comparison_df.to_csv(
    source_comparison_path,
    index=False
)

print("Source comparison saved:")
print(source_comparison_path.resolve())
print("File exists:", source_comparison_path.exists())

Source comparison saved:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Reference_Data\source_comparison_table.csv
File exists: True


### CORDIS Relationship Validation

The supporting CORDIS tables were successfully linked to the central
project table using a standardized `project_id_clean` field.

Coverage across the 23,278 Horizon Europe projects was:

- Organizations: 100%
- Horizon topics: 100%
- Legal basis: 100%
- EuroSciVoc classifications: 86.18%
- Web links: 47.60%

The lower coverage for EuroSciVoc and web links represents missing
supporting records rather than failed joins.

The supporting tables contain multiple rows per project, so they will
be aggregated to one project-level record before merging. This work
will be completed in the dedicated CORDIS cleaning and integration
notebook to avoid many-to-many row multiplication.

## Initial Data Dictionary

The initial data dictionary defines the common fields that will be
created during cleaning and integration. Source-specific field names
will be preserved in raw data, while processed datasets will use these
standardized names.

In [39]:
data_dictionary_df = pd.DataFrame([
    {
        "processed_field": "source",
        "description": "Original data source",
        "planned_type": "string",
        "NSF_source": "Constant: NSF",
        "CORDIS_source": "Constant: CORDIS",
        "OpenAlex_source": "Constant: OpenAlex",
        "required": "Yes"
    },
    {
        "processed_field": "source_record_id",
        "description": "Unique identifier from the original source",
        "planned_type": "string",
        "NSF_source": "id",
        "CORDIS_source": "id / project_id_clean",
        "OpenAlex_source": "id when publication-level data is used",
        "required": "Yes"
    },
    {
        "processed_field": "title",
        "description": "Project or publication title",
        "planned_type": "string",
        "NSF_source": "title",
        "CORDIS_source": "title",
        "OpenAlex_source": "title",
        "required": "Yes"
    },
    {
        "processed_field": "abstract",
        "description": "Project abstract, objective, or description",
        "planned_type": "string",
        "NSF_source": "abstractText",
        "CORDIS_source": "objective",
        "OpenAlex_source": "abstract_inverted_index if extracted",
        "required": "Yes for grants"
    },
    {
        "processed_field": "start_date",
        "description": "Project start date",
        "planned_type": "datetime",
        "NSF_source": "startDate",
        "CORDIS_source": "startDate",
        "OpenAlex_source": "Not applicable",
        "required": "Yes for grants"
    },
    {
        "processed_field": "end_date",
        "description": "Project end or expiration date",
        "planned_type": "datetime",
        "NSF_source": "expDate",
        "CORDIS_source": "endDate",
        "OpenAlex_source": "Not applicable",
        "required": "No"
    },
    {
        "processed_field": "record_year",
        "description": "Year used for trend analysis",
        "planned_type": "integer",
        "NSF_source": "startDate or date",
        "CORDIS_source": "startDate",
        "OpenAlex_source": "publication_year",
        "required": "Yes"
    },
    {
        "processed_field": "organisation",
        "description": "Lead or coordinating organisation",
        "planned_type": "string",
        "NSF_source": "awardeeName",
        "CORDIS_source": (
            "organization.name where role identifies coordinator"
        ),
        "OpenAlex_source": "institution when extracted",
        "required": "No"
    },
    {
        "processed_field": "country",
        "description": "Lead organisation or project country",
        "planned_type": "string",
        "NSF_source": "awardeeCountryCode",
        "CORDIS_source": "organization.country",
        "OpenAlex_source": "institution country when extracted",
        "required": "No"
    },
    {
        "processed_field": "funder",
        "description": "Funding organisation",
        "planned_type": "string",
        "NSF_source": "agency",
        "CORDIS_source": "European Commission",
        "OpenAlex_source": "Not used initially",
        "required": "Yes for grants"
    },
    {
        "processed_field": "programme",
        "description": "Funding programme, scheme, or division",
        "planned_type": "string",
        "NSF_source": (
            "fundProgramName; primaryProgram; dirAbbr; divAbbr"
        ),
        "CORDIS_source": (
            "fundingScheme; frameworkProgramme; legalBasis"
        ),
        "OpenAlex_source": "Not applicable",
        "required": "No"
    },
    {
        "processed_field": "award_amount",
        "description": "Primary reported award or contribution amount",
        "planned_type": "float",
        "NSF_source": "estimatedTotalAmt",
        "CORDIS_source": "ecMaxContribution",
        "OpenAlex_source": "Not applicable",
        "required": "No"
    },
    {
        "processed_field": "currency",
        "description": "Native currency of the funding amount",
        "planned_type": "string",
        "NSF_source": "Constant: USD",
        "CORDIS_source": "Constant: EUR",
        "OpenAlex_source": "Not applicable",
        "required": "Yes when award amount exists"
    },
    {
        "processed_field": "topic_labels",
        "description": (
            "Scientific or funding-topic classifications"
        ),
        "planned_type": "string/list",
        "NSF_source": (
            "matched_queries; programme fields; later text categories"
        ),
        "CORDIS_source": (
            "EuroSciVoc titles; Horizon topics; keywords"
        ),
        "OpenAlex_source": "GrantScopeAI topic category",
        "required": "Yes"
    },
    {
        "processed_field": "source_url",
        "description": "Link to the original source record",
        "planned_type": "string",
        "NSF_source": "Constructed from award ID",
        "CORDIS_source": "webLink.physUrl or constructed project link",
        "OpenAlex_source": "OpenAlex work URL or DOI",
        "required": "No"
    },
    {
        "processed_field": "extraction_date",
        "description": "Date the source data was acquired",
        "planned_type": "date",
        "NSF_source": "Added during acquisition",
        "CORDIS_source": "Added during processing",
        "OpenAlex_source": "Added during acquisition",
        "required": "Yes"
    }
])

display(data_dictionary_df)

,processed_field,description,planned_type,NSF_source,CORDIS_source,OpenAlex_source,required
0,source,Original data source,string,Constant: NSF,Constant: CORDIS,Constant: OpenAlex,Yes
1,source_record_id,Unique identifier from the original source,string,id,id / project_id_clean,id when publication-level data is used,Yes
2,title,Project or publication title,string,title,title,title,Yes
3,abstract,"Project abstract, objective, or description",string,abstractText,objective,abstract_inverted_index if extracted,Yes for grants
4,start_date,Project start date,datetime,startDate,startDate,Not applicable,Yes for grants
5,end_date,Project end or expiration date,datetime,expDate,endDate,Not applicable,No
6,record_year,Year used for trend analysis,integer,startDate or date,startDate,publication_year,Yes
7,organisation,Lead or coordinating organisation,string,awardeeName,organization.name where role identifies coordi...,institution when extracted,No
8,country,Lead organisation or project country,string,awardeeCountryCode,organization.country,institution country when extracted,No
9,funder,Funding organisation,string,agency,European Commission,Not used initially,Yes for grants


In [40]:
data_dictionary_path = (
    REFERENCE_DATA_DIR
    / "initial_data_dictionary.csv"
)

data_dictionary_df.to_csv(
    data_dictionary_path,
    index=False
)

print("Initial data dictionary saved:")
print(data_dictionary_path.resolve())

Initial data dictionary saved:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Reference_Data\initial_data_dictionary.csv
